### Model Evaluation Summary

The model was trained on the **Wadi Degla training dataset** and evaluated on two datasets:
1. **Wadi Degla Test Data** — to measure in-domain performance.  
2. **Full Web Data** — to assess generalization on external data.

Among the last ten epochs (121–130), **epoch 127** achieved the **best performance on the Full Web Data**.  
Its evaluation results on both datasets are summarized below:

| Model Name       | Dataset Name         | Accuracy (micro) | Precision     | Recall        | F1 Score     |
|------------------|----------------------|------------------|----------------|----------------|---------------|
| model_epoch_127  | Full Web Data        | 0.2666           | 0.3387         | 0.2268         | 0.2347        |
| model_epoch_127  | Wadi Degla Test Data | 0.8642          | 0.8730         | 0.8917         | 0.8773        |

**Observation:**  
While epoch 127 showed the best performance on the Web Data among recent epochs, the model still performs substantially better on the in-domain Wadi Degla Test Data, indicating a generalization gap between field and web image domains.


### Confidence-Based Splitting of Full Web Data

To obtain a subset of the Web Data for **expanding the training dataset**, the images were categorized based on the outputs of models from **epochs 121–130**.

First, two main groups were defined:

1. **Correctly Classified** — Images correctly classified by **at least one** of the ten models (epochs 121–130).  
2. **Misclassified Images** — Images **not correctly classified** by any of the ten models.

The **misclassified images** were then further divided into three splits according to the **confidence (predicted probability) of the actual class** — not the predicted one — as follows:

| Split Name         | Description |
|--------------------|-------------|
| **Top 30**          | Misclassified images where the probability for the actual class ranked within the **top 30** (per class). |
| **Mid (30–70)**     | Misclassified images where the probability for the actual class ranked between **30 and 70** (per class). |
| **Lowest 30**       | Misclassified images where the probability for the actual class ranked among the **lowest 30** (per class). |

In total, this process generated **four splits**: one for correctly classified images and three for misclassified images categorized by confidence level.

**Purpose:**  
This approach enables the selection of **informative and diverse samples**—particularly those with moderate to high confidence—for inclusion in the training set, helping to improve model generalization across domains.

### Extended Training Using Web Data Splits

To investigate the effect of incorporating Web Data on model generalization, two new training datasets were constructed by combining the original **Wadi Degla training set** with selected subsets from the confidence-based Web Data splits.

#### **New Training Datasets**

1. **Train Set A (Top 30 Augmented):**  
   Created by adding the **Top 30** split from the Web Data to the Wadi Degla training dataset.

2. **Train Set B (Lowest 30 Augmented):**  
   Created by adding the **Lowest 30** split from the Web Data to the Wadi Degla training dataset.

Both datasets were used to **retrain the best-performing model (epoch 127)** under the **same training hyperparameters**, but with an **enhanced data augmentation pipeline** to further improve robustness.

---

#### **Data Augmentation Pipelines**

**Previous Augmentation:**
```python
Compose([
    ToImage(),
    RandomResizedCrop(size=(224, 224), scale=(0.2, 1.0), ratio=(0.75, 1.33)),
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.2),
    RandomApply([RandomRotation(degrees=[-90.0, 90.0])]),
    ToDtype(scale=True),
    RandomApply([ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3),
                             saturation=(0.7, 1.3), hue=(-0.0278, 0.0278))]),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
````

**Enhanced Augmentation (New):**

```python
Compose([
    ToImage(),
    RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.33)),
    RandomHorizontalFlip(p=0.5),
    RandomVerticalFlip(p=0.5),
    RandomApply([RandomRotation(degrees=[-180.0, 180.0])]),
    ToDtype(scale=True),
    ColorJitter(brightness=(0.7, 1.3), contrast=(0.85, 1.3),
                saturation=(0.7, 1.3), hue=(-0.0278, 0.0278)),
    RandomApply([
        GaussianBlur(kernel_size=(3, 3), sigma=[0.1, 2.0]),
        RandomPerspective(p=1.0, distortion_scale=0.3),
        RandomErasing(p=0.5, scale=(0.02, 0.2), ratio=(0.3, 3.3), value=[0.0])
    ]),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
```

The new augmentation pipeline introduces **stronger geometric and photometric transformations** (e.g., Gaussian blur, perspective distortion, random erasing) to increase data diversity and encourage domain invariance.

---

#### **Test Datasets**

Each retrained model was evaluated on the **Wadi Degla Test Data** (shared) and on the complementary portion of the Full Web Data that excludes the split used during training.

| Retrained Model                | Test Datasets |
|--------------------------------|---------------|
| **Model trained on Train Set A** (Wadi Degla + **Top 30**)     | - Wadi Degla Test Data  <br> - Full Web **excluding Top 30** |
| **Model trained on Train Set B** (Wadi Degla + **Lowest 30**)  | - Wadi Degla Test Data  <br> - Full Web **excluding Lowest 30** |

---

**Purpose:**
This experiment aims to assess whether integrating **high-confidence** (Top 30) or **low-confidence** (Lowest 30) Web Data samples—together with stronger augmentation—can enhance the model’s **cross-domain generalization** while maintaining in-domain accuracy.

**Note:**  
We do **not** evaluate a model on the Web split that was included in its training set. This ensures the test Web subset remains unseen and provides a fair measure of generalization.

After preparing the two augmented training datasets, the model was retrained separately on each one. The starting point for both training runs was the **epoch 127 checkpoint**, previously identified as the best-performing model on web data. Training resumed from **epoch 128 to 180**, using identical hyperparameters and configurations for both datasets.

During retraining, model checkpoints were evaluated on the **Wadi Degla Test Data**, and **epoch 174** yielded the best validation performance for both datasets. Therefore, the **epoch 174** and **epoch 180** checkpoints were selected for detailed comparison to determine which augmented dataset provided better overall performance.

The comparison was conducted across the following evaluation factors:

1. **Accuracy on Wadi Degla Test Data**
2. **Accuracy on the corresponding Web Test Data**
3. **Performance on the Web Training Data used for augmentation**
4. **Recall of the lowest-performing class in the Wadi Degla Test Data**
5. **Improvement observed in the last ten classes of the Wadi Degla Test Data**, including statistical analysis of their performance changes

### **Results for Train Set A (Top 30 Augmented)**

#### **Overall Performance Summary**

| **Dataset**                  | **Epoch** | **Accuracy** | **Precision** | **Recall** | **F1-score** | **Notes**                      |
| ---------------------------- | --------- | ------------ | ------------- | ---------- | ------------ | ------------------------------ |
| **Test Data (Wadi Degla)**   | 127       | 86.42%       | 87.30%        | 89.17%     | 87.73%       | Initial checkpoint (epoch 127) |
| **Test Data (Wadi Degla)**   | 174       | 89.00%       | 89.00%        | 91.00%     | 90.00%       | **Best checkpoint**            |
| **Test Data (Wadi Degla)**   | 180       | 88.00%       | 89.00%        | 90.00%     | 89.00%       | Final epoch                    |
| **Test Data (Web Dataset)**  | 127       | 34.00%       | 41.00%        | 28.00%     | 29.00%       | Initial checkpoint (epoch 127) |
| **Test Data (Web Dataset)**  | 174       | 54.00%       | 50.00%        | 47.00%     | 47.00%       | **Best checkpoint**            |
| **Test Data (Web Dataset)**  | 180       | 55.00%       | 51.00%        | 48.00%     | 48.00%       | Final epoch                    |
| **Train Data (Web Dataset)** | 127       | 0.00%        | 0.00%         | 0.00%      | 0.00%        | Before fine-tuning             |
| **Train Data (Web Dataset)** | 174       | 89.00%       | 86.00%        | 90.00%     | 87.00%       | **Best checkpoint**            |
| **Train Data (Web Dataset)** | 180       | 90.00%       | 87.00%        | 91.00%     | 88.00%       | Final epoch                    |


### **Improvement in Lowest-Performing Classes (Wadi Degla Test Data)**

The **ten lowest-performing species** were determined based on their **recall scores** from the **base model (epoch 127)**.
Their precision, recall, and F1-scores were then tracked at **epochs 127, 174, and 180** to measure improvement resulting from retraining on the **Top 30 Augmented** dataset.

| **Species**                       | **Precision** | **Recall**  | **F1-score** | **Epoch** | **Aggregation** |
| --------------------------------- | ------------- | ----------- | ------------ | --------- | --------------- |
| *Trichodesma africanum (L.) Sm.*  | 71.90 %       | 52.10 %     | 60.42 %      | 127       | min             |
| *Heliotropium arbainense Fresen.* | 93.26 %       | 87.38 %     | 90.23 %      | 127       | max             |
| —                                 | **85.15 %**   | **73.56 %** | **78.58 %**  | **127**   | **mean**        |
| *Trichodesma africanum (L.) Sm.*  | 79.09 %       | 52.10 %     | 62.82 %      | 174       | min             |
| *Heliotropium arbainense Fresen.* | 94.61 %       | 93.69 %     | 94.15 %      | 174       | max             |
| —                                 | **88.40 %**   | **78.65 %** | **82.83 %**  | **174**   | **mean**        |
| *Trichodesma africanum (L.) Sm.*  | 81.67 %       | 58.68 %     | 68.29 %      | 180       | min             |
| *Heliotropium arbainense Fresen.* | 95.57 %       | 94.17 %     | 94.87 %      | 180       | max             |
| —                                 | **88.56 %**   | **77.69 %** | **82.48 %**  | **180**   | **mean**        |


**Interpretation:**

* The **minimum** row represents the lowest-performing class among the ten (e.g., *Trichodesma africanum*).
* The **maximum** row corresponds to the highest-performing class among the same ten (e.g., *Heliotropium arbainense*).
* The **mean** row summarizes the average performance across all ten lowest classes.

Across epochs 127 → 174 → 180, there was a clear **increase in mean recall from 73.56 % to 78.65 %**, indicating improved recognition of previously challenging species after retraining with the **Top 30 Augmented** dataset.

### **Results for Train Set B (Lowest 30 Augmented)**

| **Dataset**                  | **Epoch** | **Accuracy (micro)** | **Precision** | **Recall** | **F1-score** | **Notes**                      |
| ---------------------------- | --------- | -------------------- | ------------- | ---------- | ------------ | ------------------------------ |
| **Test Data (Wadi Degla)**   | 127       | 86.42 %              | 87.30 %       | 89.17 %    | 87.73 %      | Initial checkpoint             |
| **Test Data (Wadi Degla)**   | 174       | 88.00 %              | 89.83 %       | 89.87 %    | 89.52 %      | **Best checkpoint**            |
| **Test Data (Wadi Degla)**   | 180       | 87.54 %              | 89.37 %       | 89.62 %    | 89.12 %      | Final epoch                    |
| **Test Data (Web Dataset)**  | 127       | 33.60 %              | 38.22 %       | 27.57 %    | 28.37 %      | Initial checkpoint             |
| **Test Data (Web Dataset)**  | 174       | 64.12 %              | 57.49 %       | 57.52 %    | 56.52 %      | **Best checkpoint**            |
| **Test Data (Web Dataset)**  | 180       | 63.32 %              | 56.58 %       | 57.19 %    | 55.89 %      | Final epoch                    |
| **Train Data (Web Dataset)** | 127       | 0.00 %               | 0.00 %        | 0.00 %     | 0.00 %       | Before retraining              |
| **Train Data (Web Dataset)** | 174       | 67.64 %              | 67.17 %       | 67.21 %    | 64.73 %      | **Best checkpoint**            |
| **Train Data (Web Dataset)** | 180       | 68.97 %              | 67.82 %       | 68.69 %    | 65.99 %      | Final epoch                    |

---

### **Improvement in Lowest-Performing Classes (Wadi Degla Test Data)**

The **ten lowest-performing species** were identified based on their **recall scores** from the **base model (epoch 127)**.
Their precision, recall, and F1-scores were tracked at **epochs 127, 174, and 180** to assess improvement resulting from retraining on the **Lowest 30 Augmented** dataset.

| **Species**                        | **Precision** | **Recall**  | **F1-score** | **Epoch** | **Aggregation** |
| ---------------------------------- | ------------- | ----------- | ------------ | --------- | --------------- |
| *Trichodesma africanum (L.) Sm.*   | 71.90 %       | 52.10 %     | 60.42 %      | 127       | min             |
| *Heliotropium arbainense Fresen.*  | 93.26 %       | 87.38 %     | 90.23 %      | 127       | max             |
| —                                  | **85.15 %**   | **73.56 %** | **78.58 %**  | **127**   | **mean**        |
| *Tamarix nilotica (Ehrenb.) Bunge* | 94.95 %       | 64.69 %     | 76.95 %      | 174       | min             |
| *Heliotropium arbainense Fresen.*  | 95.43 %       | 91.26 %     | 93.30 %      | 174       | max             |
| —                                  | **87.99 %**   | **77.63 %** | **82.19 %**  | **174**   | **mean**        |
| *Trichodesma africanum (L.) Sm.*   | 80.60 %       | 64.67 %     | 71.76 %      | 180       | min             |
| *Heliotropium arbainense Fresen.*  | 94.03 %       | 91.75 %     | 92.87 %      | 180       | max             |
| —                                  | **88.08 %**   | **76.74 %** | **81.81 %**  | **180**   | **mean**        |

---

**Interpretation:**

* The **mean recall** across the ten lowest classes increased from **73.56 % (epoch 127)** to **77.63 % (epoch 174)** and **76.74 % (epoch 180)**.
* This suggests that including **low-confidence (Lowest 30)** samples from the Web Data improved recognition of previously underperforming species while maintaining stable overall accuracy on the Wadi Degla test set.